# **Spam Email Classification**

Spam email is a broad term used to describe unsolicited messages including: marketing, scams, or malware. These are often redundant, or could even potentially pose security risks to the recipient, and filtering is a common option used in removing spam from inboxes. 

This project will explore the historical data to develop a machine learning model that is capable of flagging spam emails based on a pre-defined set of attributes. This study will focus more on using reputable dataset and classification techniques to train a machine learning model rather than producing a deployable system that can be used in today's real-world environment.

This project uses the Spambase dataset from the UCI Machine Learning Repository.

**Citation**: <br>
Hopkins, M., Reeber, E., Forman, G., & Suermondt, J. (1999). <br>
Spambase [Dataset]. UCI Machine Learning Repository. <br>
https://archive.ics.uci.edu/dataset/94/spambase

**License**: CC BY 4.0

## **Table of Contents**

## 1. **Problem Statement**

Effective spam detection requires a clear understanding of the features that distinguish spam emails from legitimate messages. Through identification and analysis, the risk of misclassification can be reduced, therefore improving the model performance. This study uses the Spambase dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/), a reputable public dataset used for email classification tasks. This dataset contains over 50 fields which can help contextualise the definition of a spam email. Furthermore, each record has been pre-assigned classification labels with approximately ~7% misclassification, making the labels reasonably reliable for modelling and evaluating classification.

### 1.1 - The Aim

***To develop a machine learning model capable of classifying spam and non-spam emails, achieving up to a 90% accuracy while maintaining a balanced performance accross precision and recall.***

### 1.2 - Objectives

**Main Objectives**
- Clean the dataset to improve the reliability of data used to train machine learning models, and document any changes.
    - Remove redundancy such as duplicates.
    - Conduct sense checks to ensure all fields are valid.
    - Fix any consistency issues within the data.
    - Verify the presence of missing or null values and handle if necessary.
    - Consider if data parsing would enrich the dataset.
    - Determine if outliers are present within the dataset and if so, handle it accordingly.
- Preliminary analysis to understand the shape of the data.
    - Check for class imbalances and address it if present.
    - Plot the distributions of features - Only highlight the most noteworthy features as there are a lot.
    - Assess feature distributions for skewness and consider transformations if needed.
    - Determine if the data needs to be normalised and handle it when applicable.
    - List out summary statistics.
- Conduct exploratory data analysis to evaluate the importance of each feature in classifying emails.
    - Correlation Matrix
    - Random Forest Feature Importances
    - Univariate Feature Selection (SelectKBest)
- Train classification models.
    - Split dataset into a training and testing set - Consider k-fold cross-validation for training.
    - Logistic Regression
    - Random Forest
    - Naive Bayes
- Evaluate the performance of each classification model.
    - Accuracy
    - Precision
    - Recall
    - F1 Score
    - Confusion Matrix
    - Establish a baseline model to provide a reference point for evaluating model performance.

**Additional Objectives**
- Conduct feature engineering to develop a ```Spam Score``` that measures the likelihood for spam flagging.
    - Create a calculated score based on available features.
    - Consider omitting features that have no impact on classification to reduce redundancy.
    - Apply weights according to feature importance.
- Evaluate the ```Spam Score``` to determine the viability of this calculated measure compared to standard machine learning models.
    - Train classification models using the ```Spam Score```.
    - Measure the performance of the models using accuracy score, F1 score, and precision and recall, then compare it against previous models.

### 1.3 - Methodology

**Excel** will be used to conduct the initial data inspection used to handle basic cleaning tasks, such as removing redundancy, and performing sense checks. Subsequently, **Python** will be utilised for more advanced data pre-processing and analysis, including outlier detection, normalisation, and data transformation.

Modelling of machine learning models and analysis will be done using Python libraries such as **pandas**, **sckit-learn**, **matplotlib**, and **seaborn**. From our defined list of objectives, classification techniques will be conducted, which will applied alongside feature analysis methods to evaluate the contribution of individual features to spam detection. These models will be trained using training and testing data which may undergo class rebalancing and k-fold cross-validation if necessary, to enhance the reliability of outcomes.

Multiple classification models will be trained to distinguish spam and non-spam emails. Each model's performance will then be evaluated using accuracy, precision, and recall techniques, with the intent of achieving a strong overall classification performance whilst minimising misclassification. If necessary, ensemble learning can be explored to further enhance model performance.

### 1.4 - Scope & Limitations

From the aim, two sets of objectives were derived. 
- A set of main objectives, representing core tasks required to complete the study.
- A set of additional objectives, representing optional tasks that enhance the depth and reliability of analysis.

These objectives were split into two parts for task prioritisation, ensuring that the essential components are completed within the alloted timeframe with room for development if time permits. The primary focus is ultimately to deliver a minimum viable product before extending the study.

This leads into the limitations:
- Time constraints.
- Limited domain knowledge.
- Limited technical knowledge.
- Use of a static dataset.

Of the limitations, the following were considered:

**Time constraints**
- Necessitates the prioritisation of core tasks over optional objectives.
- Limit the overall scope to ensure timely completion.
- Optional tasks should only be persued after core tasks have been achieved.

**Limited domain knowledge**
- Reduces the interpretability of this study for real-world relevance.
- Study should be treated as research rather than real-world application.

**Limited technical knowledge**
- Balance familiar modelling techniques (Random Forest, Logistic regression) and unfamiliar techniques (Naive Bayes).
- Encourages self-development and exploration, whilst maintaining a managable level of complexity.
- Reduces the amount of time spent learning new concepts and techniques.

**Use of a static dataset**
- The dataset is historical and may not reflect modern spam characteristics.
- Spam has becoming increasingly sophisticated, reducing the real-world revelance of this dataset.
- Project was framed as a preliminary study used for exploration of fundamentals.
- Project can be delved further for real-world application if updated with recent data, which can be compared to the findings of this study to determine changes over time.


## 2. **Libraries**

In [12]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy.stats import zscore

## 3. **Data Pre-processing**

### 2.1 - Fields

**Field name rules:**
- Lower casing words.
- Underscores used for spaces.
- Field names with '_pct' are percentages.

**Field definitions:**
- *word_freq_(WORD)*
    - Frequency a word appears within an email.
    - Always calculated as a percentage of total words.
- *char_freq_(CHAR)*
    - Frequency a character appears within an email.
    - Always calculated as a percentage of total words.
- *captial_run_length_average*
    - Average length of sequences of consecutive capital letters.
- *captial_run_length_longest*
    - Maximum length of sequences of consecutive capital letters.
- *captial_run_length_total*
    - Total number of capital letters within an email.
- *is_spam*
    - 1 is spam.
    - 0 is ham (legitimate email).

Percent Encoded Charset:
- char_freq_**%3B** - **;**
- char_freq_**%28** - **(**
- char_freq_**%5B** - **[**
- char_freq_**%21** - **!**
- char_freq_**%24** - **$**
- char_freq_**%23** - **#**

### 2.2 - Cleaning the Dataset

**Dataset changes on Excel:**
- Original copy for rollback.
    - *What was done*: 
        - Made a backup of the original dataset.
        - Converted the dataset into a xlsx file.
    - **Version number**: v.1.0
- Feature renaming.
    - *What was done*: 
        - Added *'_pct'* at the end of field names that are calculated as a percentage.
        - Renamed *'class'* into *'is_spam'* for clarity of binary classifications.
    - **Version number**: v1.1

<br>

**Data Insights:**
- Dataset shape
    - **4601** rows.
    - Spread across **58** columns.
- **391** duplicates found on Excel 'remove duplicates' feature.
    - Kept in the dataset.
    - Avoids underrepresenting patterns of spam or non-spam email.
    - Maintains data distribution.
- Missing value checks.
    - No blanks detected using COUNTBLANK.
    - Zero values may represent absence of a feature rather than missing data.
- Sense checks.
    - Extreme values defined by human inference, using 20% as a threshold for relatively high values.
    - Extreme values found in the following fields:
        - *word_freq_3d_pct*
        - *word_freq_george_pct*
        - *word_freq_hp_pct*
        - *word_freq_project_pct*
        - *word_freq_re_pct*
        - *word_freq_edu_pct*
        - *char_freq_%21_pct*
        - *char_freq_%23_pct*
        - *capital_run_length_average*
        - *capital_run_length_longest*
        - *capital_run_length_total*
    - Further testing required will be conducted on Python using z-score or IQR to determine actual outliers based on mathematical testing.
- Value validation.
    - Data only contains numeric fields.
        - Performed a COUNT on each field expected to only contain numeric values.
        - Returned **4601** for each field (same number of rows), suggesting pure numeric values in each column.
    - No text fields therefore no spell checks or text formatting required.
    - Range checks.
        - Percentages do not exceed 100%.
        - Only two values are present for 'is_spam', 0 (**2788** values) or 1 (**1813** values).
            - Moderate class imbalance should be noted for models (**61%** non-spam to **39%** spam).
- Standardisation.
    - *'_pct'* was added to each percentage field for feature renaming.
    - No metric fields requiring standardisation at this stage.
- Data parsing.
    - Will be considered for optional creation of *'spam_score'*.

**Further data preprocessing will be conducted using Python for more thorough checks e.g.**
- Data scaling
- Outlier detection
- Train/test split
- Feature selection
- Distribution

### 2.3 - Dataset

In [2]:
data_Path = r'Dataset\spambase_v1.1_feature_renaming.csv'
data = pd.read_csv(data_Path)

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make_pct          4601 non-null   float64
 1   word_freq_address_pct       4601 non-null   float64
 2   word_freq_all_pct           4601 non-null   float64
 3   word_freq_3d_pct            4601 non-null   float64
 4   word_freq_our_pct           4601 non-null   float64
 5   word_freq_over_pct          4601 non-null   float64
 6   word_freq_remove_pct        4601 non-null   float64
 7   word_freq_internet_pct      4601 non-null   float64
 8   word_freq_order_pct         4601 non-null   float64
 9   word_freq_mail_pct          4601 non-null   float64
 10  word_freq_receive_pct       4601 non-null   float64
 11  word_freq_will_pct          4601 non-null   float64
 12  word_freq_people_pct        4601 non-null   float64
 13  word_freq_report_pct        4601 

In [21]:
data.describe()

,word_freq_make_pct,word_freq_address_pct,word_freq_all_pct,word_freq_3d_pct,word_freq_our_pct,word_freq_over_pct,word_freq_remove_pct,word_freq_internet_pct,word_freq_order_pct,word_freq_mail_pct,...,char_freq_%3B_pct,char_freq_%28_pct,char_freq_%5B_pct,char_freq_%21_pct,char_freq_%24_pct,char_freq_%23_pct,capital_run_length_average,capital_run_length_longest,capital_run_length_total,is_spam
count,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,...,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000,4601.000000
mean,0.104553,0.213015,0.280656,0.065425,0.312223,0.095901,0.114208,0.105295,0.090067,0.239413,...,0.038575,0.139030,0.016976,0.269071,0.075811,0.044238,5.191515,52.172789,283.289285,0.394045
std,0.305358,1.290575,0.504143,1.395151,0.672513,0.273824,0.391441,0.401071,0.278616,0.644755,...,0.243471,0.270355,0.109394,0.815672,0.245882,0.429342,31.729449,194.891310,606.347851,0.488698
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.588000,6.000000,35.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.065000,0.000000,0.000000,0.000000,0.000000,2.276000,15.000000,95.000000,0.000000
75%,0.000000,0.000000,0.420000,0.000000,0.380000,0.000000,0.000000,0.000000,0.000000,0.160000,...,0.000000,0.188000,0.000000,0.315000,0.052000,0.000000,3.706000,43.000000,266.000000,1.000000
max,4.540000,14.280000,5.100000,42.810000,10.000000,5.880000,7.270000,11.110000,5.260000,18.180000,...,4.385000,9.752000,4.081000,32.478000,6.003000,19.829000,1102.500000,9989.000000,15841.000000,1.000000


#### 2.3.1 - Checking for Outliers

**Using z-score**

In [ ]:
outlier_Std = 3                         #Where 3 is default, any value exceeding 3 standard deviations will be classified as an outlier.
data_Columns = data.columns.values      #Columns in the dataset.
zscore_Feature_Threshold = 3            #Maximum number of features that can be flagged for outliers.

#print(len(data_Columns))

In [ ]:
#Calculate the z-scores for each feature.
data_ZScore = pd.DataFrame(zscore(data[data_Columns], nan_policy='omit'), columns=data_Columns, index=data.index)

#Flag rows where at least three (zscore_Feature_Threshold) feature exceeds z-score threshold.
zscore_Count = (abs(data_ZScore) > outlier_Std).sum(axis=1)
data_Outliers = data[zscore_Count > zscore_Feature_Threshold]

#print(data_Outliers)
print(f'Number of detected outliers from {outlier_Std} standard deviations: {len(data_Outliers)}')

Number of detected outliers from 3 standard deviations: 125
